# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available Record Sets, Fields, and their `@id` values from the Croissant schema. This helps identify what structured tables (Record Sets) are available in the dataset and what each contains.

All references use the canonical `@id` as required.

In [ ]:
# List available Record Sets
print("--- Available Record Sets (@id and name) ---")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name}")

# Choose one Record Set for demonstration (the main table)
# Typically, the first Record Set is the main dataset. Let's display its fields by @id:
if not record_sets:
    raise ValueError("No record sets found in the dataset schema.")
main_rs = record_sets[0]  # Use the first available

print(f"\n--- Fields in Record Set '@id': {main_rs.id} ({main_rs.name}) ---")
for field in main_rs.fields:
    print(f"Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the above overview. All extractions reference entities using their Croissant canonical `@id`s.

In [ ]:
# List all record set @ids for reference
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load each record set into a DataFrame
for rset in record_sets:
    records = list(dataset.records(record_set=rset.id))
    dataframes[rset.id] = pd.DataFrame(records)

# Display available columns in the main record set
print(f"Main record set @id: {main_rs.id}")
print("Record set columns:", dataframes[main_rs.id].columns.tolist())

# Preview first 5 records
dataframes[main_rs.id].head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (by `@id`) and demonstrate analysis: filtering, normalization, and grouping. Make sure to refer to fields using their Croissant `@id`.

*Replace the `numeric_field_id` and `group_field_id` below by the actual `@id`s found in the overview.*

In [ ]:
# Choose a numeric field (@id) and group field (@id)
# (You may need to check the output of section 2)
# Example placeholder values below--REPLACE with real ones based on the printed fields
# For demonstration, here we choose the first field with an integer/float type, and group by another field

# Identify a numeric field
possible_numeric_fields = [field.id for field in main_rs.fields if getattr(field, 'data_type', '').lower() in ('integer', 'float', 'number')]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # Fallback if not annotated: try to infer from column name
    numeric_field_id = dataframes[main_rs.id].select_dtypes(include=[float, int]).columns[0]

# Identify a possible group field
possible_group_fields = [field.id for field in main_rs.fields if getattr(field, 'data_type', '').lower() == 'text']
if possible_group_fields:
    group_field_id = possible_group_fields[0]
else:
    group_field_id = None

print(f"Using numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Grouping by field: {group_field_id}")

# Filter records where the numeric field > some threshold (e.g., 10)
threshold = 10
filtered_df = dataframes[main_rs.id][dataframes[main_rs.id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (first 5 rows):")
print(filtered_df.head())

# Normalize the numeric field (standard score)
filtered_df[numeric_field_id + "_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records (first 5 rows):")
print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data (mean of {numeric_field_id}) by {group_field_id} (first 5 groups):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field or show relationships between two key fields, using the DataFrame extracted from the record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the filtered numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field present, boxplot
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to explore and analyze a hierarchical FAIR^2 dataset using `mlcroissant`, referencing all Record Sets and Fields by their canonical `@id`. Further analyses may be performed based on biological questions or downstream research tasks, leveraging the dataset's rich structure.

For further information, consult the [Croissant specification](https://mlcommons.org/croissant/) and the dataset [FAIR^2 landing page](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).